# Generate Submission v2

**Purpose:** produce `submission_v2.csv`, using the new best model found in this project —
gender-split logistic regression with 6 features (the original 5 from notebook 13, plus
`ClusterRankDiff` from notebook 18), which scored 0.1669 average walk-forward Brier, an
improvement on notebook 13's 0.1675 (which produced `submission.csv`, scored 0.1302747 on the
real Kaggle leaderboard).

**How this differs from notebook 16:** same production approach — retrain on every historical
tournament game through 2025, no held-out fold, since 2026 has no precedent to hold back. The only
change is the extra feature. That means this notebook also needs to redo notebook 18's clustering
step (fit separately per gender on the full history, including 2026 team-seasons) so that a
`ClusterRank` exists for every 2026 tournament team, not just historical ones.

## 1. Load data

Same files as notebook 16, plus the submission template.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)

DATA_DIR = Path("../data")

m_tourney = pd.read_csv(DATA_DIR / "MNCAATourneyCompactResults.csv")
w_tourney = pd.read_csv(DATA_DIR / "WNCAATourneyCompactResults.csv")
m_seeds = pd.read_csv(DATA_DIR / "MNCAATourneySeeds.csv")
w_seeds = pd.read_csv(DATA_DIR / "WNCAATourneySeeds.csv")
m_reg = pd.read_csv(DATA_DIR / "MRegularSeasonCompactResults.csv")
w_reg = pd.read_csv(DATA_DIR / "WRegularSeasonCompactResults.csv")
m_conf_teams = pd.read_csv(DATA_DIR / "MTeamConferences.csv")
w_conf_teams = pd.read_csv(DATA_DIR / "WTeamConferences.csv")
sample_sub = pd.read_csv(DATA_DIR / "SampleSubmissionStage2.csv")

print(f"historical men's tourney games (training labels): {len(m_tourney):,}")
print(f"historical women's tourney games (training labels): {len(w_tourney):,}")
print(f"men's 2026 seeds: {(m_seeds['Season'] == 2026).sum()}, women's 2026 seeds: {(w_seeds['Season'] == 2026).sum()}")
print(f"submission rows to predict: {len(sample_sub):,}")


historical men's tourney games (training labels): 2,585
historical women's tourney games (training labels): 1,717
men's 2026 seeds: 68, women's 2026 seeds: 68
submission rows to predict: 132,133


## 2. Whole-season team features, Elo, seeds, and conference strength (unchanged from notebooks 13/16)

In [2]:
def team_season_stats(reg):
    won = reg[["Season", "WTeamID", "WScore", "LScore"]].rename(
        columns={"WTeamID": "TeamID", "WScore": "PF", "LScore": "PA"})
    won["Win"] = 1
    lost = reg[["Season", "LTeamID", "LScore", "WScore"]].rename(
        columns={"LTeamID": "TeamID", "LScore": "PF", "WScore": "PA"})
    lost["Win"] = 0
    games = pd.concat([won, lost], ignore_index=True)
    stats = games.groupby(["Season", "TeamID"]).agg(
        GamesPlayed=("Win", "size"), WinPct=("Win", "mean"),
        AvgPF=("PF", "mean"), AvgPA=("PA", "mean"),
    ).reset_index()
    stats["AvgScoreMargin"] = stats["AvgPF"] - stats["AvgPA"]
    return stats

team_stats = pd.concat([team_season_stats(m_reg), team_season_stats(w_reg)], ignore_index=True)

def seed_num(s):
    return int("".join(ch for ch in s if ch.isdigit()))

m_seeds["SeedNum"] = m_seeds["Seed"].apply(seed_num)
w_seeds["SeedNum"] = w_seeds["Seed"].apply(seed_num)
m_seeds["Gender"], w_seeds["Gender"] = "M", "W"
seeds = pd.concat([m_seeds[["Season", "TeamID", "SeedNum", "Gender"]], w_seeds[["Season", "TeamID", "SeedNum", "Gender"]]], ignore_index=True)
seed_idx = seeds.set_index(["Season", "TeamID"])["SeedNum"]

def compute_elo(compact_df, k=20, home_adv=100, mean_reversion=0.75):
    elo = {}
    elo_by_season_end = {}
    for season in sorted(compact_df["Season"].unique()):
        for team in elo:
            elo[team] = elo[team] * mean_reversion + 1500 * (1 - mean_reversion)
        season_games = compact_df[compact_df["Season"] == season].sort_values("DayNum")
        for game in season_games.itertuples(index=False):
            w, l = game.WTeamID, game.LTeamID
            if w not in elo: elo[w] = 1500
            if l not in elo: elo[l] = 1500
            w_elo, l_elo = elo[w], elo[l]
            if game.WLoc == "H": w_elo += home_adv
            elif game.WLoc == "A": l_elo += home_adv
            w_exp = 1 / (1 + 10 ** ((l_elo - w_elo) / 400))
            mov = game.WScore - game.LScore
            mov_mult = np.log(max(mov, 1) + 1) * (2.2 / ((w_elo - l_elo) * 0.001 + 2.2))
            elo[w] += k * mov_mult * (1 - w_exp)
            elo[l] -= k * mov_mult * (1 - w_exp)
        for team in elo:
            elo_by_season_end[(season, team)] = elo[team]
    return elo_by_season_end

m_elo = compute_elo(m_reg)
w_elo = compute_elo(w_reg)
elo_idx = pd.Series({**m_elo, **w_elo})
elo_idx.index = pd.MultiIndex.from_tuples(elo_idx.index, names=["Season", "TeamID"])

conf_teams = pd.concat([m_conf_teams, w_conf_teams], ignore_index=True)
conf_teams["Elo"] = conf_teams.apply(lambda r: elo_idx.get((r["Season"], r["TeamID"]), np.nan), axis=1)
conf_teams_valid = conf_teams.dropna(subset=["Elo"])
conf_strength = conf_teams_valid.groupby(["Season", "ConfAbbrev"])["Elo"].mean().rename("ConfStrength")
team_conf_strength = conf_teams_valid.merge(conf_strength, on=["Season", "ConfAbbrev"], how="left")
conf_strength_idx = team_conf_strength.set_index(["Season", "TeamID"])["ConfStrength"]

print(f"team-season stats: {len(team_stats):,}, seeds: {len(seeds):,}, Elo entries: {len(elo_idx):,}, conf-strength entries: {len(conf_strength_idx):,}")
print(f"2026 teams with an Elo rating: {elo_idx.loc[2026].shape[0] if 2026 in elo_idx.index.get_level_values('Season') else 0}")


team-season stats: 23,604, seeds: 4,506, Elo entries: 24,158, conf-strength entries: 23,606
2026 teams with an Elo rating: 751


## 3. Cluster separately per gender, on the full history including 2026 (unchanged from notebook 18)

Same hand-written k-means and elbow method as notebook 18, run on every team-season with a
complete feature set (SeedNum, WinPct, AvgScoreMargin, Elo, ConfStrength) — including 2026, so
that every 2026 tournament team ends up with a `ClusterRank`.

In [3]:
team_season = seeds.merge(team_stats[["Season", "TeamID", "WinPct", "AvgScoreMargin"]], on=["Season", "TeamID"], how="left")
team_season["Elo"] = team_season.apply(lambda r: elo_idx.get((r["Season"], r["TeamID"]), np.nan), axis=1)
team_season = team_season.merge(team_conf_strength[["Season", "TeamID", "ConfStrength"]], on=["Season", "TeamID"], how="left")

FEATURES_CLUSTER = ["SeedNum", "WinPct", "AvgScoreMargin", "Elo", "ConfStrength"]
team_season = team_season.dropna(subset=FEATURES_CLUSTER).reset_index(drop=True)
print(f"team-season rows with complete cluster features: {len(team_season):,}")
print(f"2026 team-seasons with complete cluster features: {(team_season['Season']==2026).sum()}")

def kmeans(X, k, n_iter=200, seed=42):
    rng = np.random.default_rng(seed)
    n = X.shape[0]
    centers = X[rng.choice(n, size=k, replace=False)].copy()
    labels = np.zeros(n, dtype=int)
    for it in range(n_iter):
        dists = np.linalg.norm(X[:, None, :] - centers[None, :, :], axis=2)
        new_labels = np.argmin(dists, axis=1)
        if np.array_equal(new_labels, labels) and it > 0:
            break
        labels = new_labels
        for j in range(k):
            if np.any(labels == j):
                centers[j] = X[labels == j].mean(axis=0)
    dists = np.linalg.norm(X[:, None, :] - centers[None, :, :], axis=2)
    inertia = np.sum(np.min(dists, axis=1) ** 2)
    return labels, centers, inertia

def choose_k_and_cluster(X_raw, k_range=range(2, 9)):
    X = (X_raw - X_raw.mean(axis=0)) / X_raw.std(axis=0)
    inertias = []
    for k in k_range:
        _, _, inertia = kmeans(X, k)
        inertias.append(inertia)
    pct_improvement = [(inertias[i-1] - inertias[i]) / inertias[i-1] for i in range(1, len(inertias))]
    chosen_k = next((list(k_range)[i] for i, p in enumerate(pct_improvement) if p < 0.10), list(k_range)[-1])
    labels, centers, _ = kmeans(X, chosen_k)
    return labels, chosen_k, inertias

cluster_assignments = pd.Series(index=team_season.index, dtype=int)
chosen_ks = {}
for gender in ["M", "W"]:
    mask = team_season["Gender"] == gender
    X_raw = team_season.loc[mask, FEATURES_CLUSTER].values
    labels, chosen_k, inertias = choose_k_and_cluster(X_raw)
    cluster_assignments.loc[mask] = labels
    chosen_ks[gender] = chosen_k
    print(f"{gender}: chosen k = {chosen_k}")

team_season["ClusterRaw"] = cluster_assignments.astype(int)

team_season["ClusterRank"] = np.nan
for gender in ["M", "W"]:
    mask = team_season["Gender"] == gender
    cluster_elo = team_season[mask].groupby("ClusterRaw")["Elo"].mean().sort_values(ascending=False)
    rank_map = {cid: rank + 1 for rank, cid in enumerate(cluster_elo.index)}
    team_season.loc[mask, "ClusterRank"] = team_season.loc[mask, "ClusterRaw"].map(rank_map)

cluster_rank_idx = team_season.set_index(["Season", "TeamID"])["ClusterRank"]
print(f"2026 teams with a ClusterRank: {team_season[team_season['Season']==2026]['ClusterRank'].notna().sum()}")


team-season rows with complete cluster features: 4,506
2026 team-seasons with complete cluster features: 136
M: chosen k = 5
W: chosen k = 8
2026 teams with a ClusterRank: 136


## 4. Build the training matchup table (historical tournament games only)

Same as notebook 16, with `ClusterRankDiff` added as a 6th feature.

In [4]:
def build_matchups(tourney, gender):
    df = tourney.copy()
    df["Team1"] = df[["WTeamID", "LTeamID"]].min(axis=1)
    df["Team2"] = df[["WTeamID", "LTeamID"]].max(axis=1)
    df["Label"] = (df["WTeamID"] == df["Team1"]).astype(int)
    df["Gender"] = gender
    return df[["Season", "Team1", "Team2", "Label", "Gender"]]

def add_features(matchups):
    matchups = matchups.copy()
    matchups["Seed1"] = matchups.apply(lambda r: seed_idx.get((r["Season"], r["Team1"]), np.nan), axis=1)
    matchups["Seed2"] = matchups.apply(lambda r: seed_idx.get((r["Season"], r["Team2"]), np.nan), axis=1)
    matchups["SeedDiff"] = matchups["Seed2"] - matchups["Seed1"]
    matchups["Elo1"] = matchups.apply(lambda r: elo_idx.get((r["Season"], r["Team1"]), np.nan), axis=1)
    matchups["Elo2"] = matchups.apply(lambda r: elo_idx.get((r["Season"], r["Team2"]), np.nan), axis=1)
    matchups["EloDiff"] = matchups["Elo1"] - matchups["Elo2"]
    matchups["ConfStrength1"] = matchups.apply(lambda r: conf_strength_idx.get((r["Season"], r["Team1"]), np.nan), axis=1)
    matchups["ConfStrength2"] = matchups.apply(lambda r: conf_strength_idx.get((r["Season"], r["Team2"]), np.nan), axis=1)
    matchups["ConfStrengthDiff"] = matchups["ConfStrength1"] - matchups["ConfStrength2"]
    matchups["ClusterRank1"] = matchups.apply(lambda r: cluster_rank_idx.get((r["Season"], r["Team1"]), np.nan), axis=1)
    matchups["ClusterRank2"] = matchups.apply(lambda r: cluster_rank_idx.get((r["Season"], r["Team2"]), np.nan), axis=1)
    matchups["ClusterRankDiff"] = matchups["ClusterRank1"] - matchups["ClusterRank2"]
    for team_col, suffix in [("Team1", "1"), ("Team2", "2")]:
        joined = matchups.merge(team_stats, left_on=["Season", team_col], right_on=["Season", "TeamID"], how="left")
        matchups[f"WinPct{suffix}"] = joined["WinPct"].values
        matchups[f"AvgMargin{suffix}"] = joined["AvgScoreMargin"].values
    matchups["WinPctDiff"] = matchups["WinPct1"] - matchups["WinPct2"]
    matchups["AvgMarginDiff"] = matchups["AvgMargin1"] - matchups["AvgMargin2"]
    return matchups

FEATURES = ["SeedDiff", "WinPctDiff", "AvgMarginDiff", "EloDiff", "ConfStrengthDiff", "ClusterRankDiff"]

training_matchups = add_features(
    pd.concat([build_matchups(m_tourney, "M"), build_matchups(w_tourney, "W")], ignore_index=True)
)
training_matchups = training_matchups.dropna(subset=FEATURES).reset_index(drop=True)
print(f"training rows: {len(training_matchups):,} (men's: {(training_matchups['Gender']=='M').sum():,}, women's: {(training_matchups['Gender']=='W').sum():,})")


training rows: 4,302 (men's: 2,585, women's: 1,717)


## 5. Train the final models on all historical data (6 features)

In [5]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -30, 30)))

def fit_logreg(X, y, l2=1.0, n_iter=50, tol=1e-8):
    n, p = X.shape
    Xb = np.hstack([np.ones((n, 1)), X])
    w = np.zeros(p + 1)
    reg = np.eye(p + 1) * l2
    reg[0, 0] = 0.0
    for _ in range(n_iter):
        pr = sigmoid(Xb @ w)
        grad = Xb.T @ (pr - y) + reg @ w
        W = np.clip(pr * (1 - pr), 1e-6, None)
        H = Xb.T @ (Xb * W[:, None]) + reg
        step = np.linalg.solve(H, grad)
        w_new = w - step
        if np.max(np.abs(w_new - w)) < tol:
            w = w_new
            break
        w = w_new
    return w

def predict_proba(X, w):
    Xb = np.hstack([np.ones((X.shape[0], 1)), X])
    return sigmoid(Xb @ w)

men_train = training_matchups[training_matchups["Gender"] == "M"]
women_train = training_matchups[training_matchups["Gender"] == "W"]

w_men_final = fit_logreg(men_train[FEATURES].values, men_train["Label"].values)
w_women_final = fit_logreg(women_train[FEATURES].values, women_train["Label"].values)

print("final men's model coefficients (intercept, then SeedDiff/WinPctDiff/AvgMarginDiff/EloDiff/ConfStrengthDiff/ClusterRankDiff):")
print(np.round(w_men_final, 5))
print("final women's model coefficients:")
print(np.round(w_women_final, 5))


final men's model coefficients (intercept, then SeedDiff/WinPctDiff/AvgMarginDiff/EloDiff/ConfStrengthDiff/ClusterRankDiff):
[-0.01555  0.05132 -1.02071  0.06149  0.00484  0.0023   0.11912]
final women's model coefficients:
[ 0.09602  0.1135  -1.53798  0.06726  0.00466  0.00339  0.06381]


## 6. Compute 2026 features and predict for every submission row

Same convention as notebook 16 — `Team1` is always the lower TeamID, men's model for TeamIDs
under 2000, women's model for 3000+, and a neutral 0.5 default for any row missing a required
feature (almost always because one of the two teams didn't make the 2026 tournament, and so has
no seed, and by extension no cluster rank for this season).

In [6]:
ids = sample_sub["ID"].str.split("_", expand=True).astype(int)
ids.columns = ["Season", "Team1", "Team2"]
sub = pd.concat([sample_sub[["ID"]], ids], axis=1)

sub["Seed1"] = sub.apply(lambda r: seed_idx.get((r["Season"], r["Team1"]), np.nan), axis=1)
sub["Seed2"] = sub.apply(lambda r: seed_idx.get((r["Season"], r["Team2"]), np.nan), axis=1)
sub["SeedDiff"] = sub["Seed2"] - sub["Seed1"]
sub["Elo1"] = sub.apply(lambda r: elo_idx.get((r["Season"], r["Team1"]), np.nan), axis=1)
sub["Elo2"] = sub.apply(lambda r: elo_idx.get((r["Season"], r["Team2"]), np.nan), axis=1)
sub["EloDiff"] = sub["Elo1"] - sub["Elo2"]
sub["ConfStrength1"] = sub.apply(lambda r: conf_strength_idx.get((r["Season"], r["Team1"]), np.nan), axis=1)
sub["ConfStrength2"] = sub.apply(lambda r: conf_strength_idx.get((r["Season"], r["Team2"]), np.nan), axis=1)
sub["ConfStrengthDiff"] = sub["ConfStrength1"] - sub["ConfStrength2"]
sub["ClusterRank1"] = sub.apply(lambda r: cluster_rank_idx.get((r["Season"], r["Team1"]), np.nan), axis=1)
sub["ClusterRank2"] = sub.apply(lambda r: cluster_rank_idx.get((r["Season"], r["Team2"]), np.nan), axis=1)
sub["ClusterRankDiff"] = sub["ClusterRank1"] - sub["ClusterRank2"]
for team_col, suffix in [("Team1", "1"), ("Team2", "2")]:
    joined = sub.merge(team_stats, left_on=["Season", team_col], right_on=["Season", "TeamID"], how="left")
    sub[f"WinPct{suffix}"] = joined["WinPct"].values
    sub[f"AvgMargin{suffix}"] = joined["AvgScoreMargin"].values
sub["WinPctDiff"] = sub["WinPct1"] - sub["WinPct2"]
sub["AvgMarginDiff"] = sub["AvgMargin1"] - sub["AvgMargin2"]

has_all_features = sub[FEATURES].notna().all(axis=1)
is_men = sub["Team1"] < 2000

pred = np.full(len(sub), 0.5)
men_mask = has_all_features & is_men
women_mask = has_all_features & ~is_men
pred[men_mask.values] = predict_proba(sub.loc[men_mask, FEATURES].values, w_men_final)
pred[women_mask.values] = predict_proba(sub.loc[women_mask, FEATURES].values, w_women_final)

sub["Pred"] = pred

print(f"rows with a real (non-default) prediction: {has_all_features.sum():,} of {len(sub):,}")
print(f"  men's: {men_mask.sum():,}, women's: {women_mask.sum():,}")
print(f"prediction range: {sub['Pred'].min():.4f} to {sub['Pred'].max():.4f}")


rows with a real (non-default) prediction: 4,556 of 132,133
  men's: 2,278, women's: 2,278
prediction range: 0.0008 to 0.9996


## 7. Sanity checks before saving

Same checks as notebook 16, plus a direct comparison against `submission.csv` (the notebook
13/16 model) to see how much the added feature actually moves individual predictions.

In [7]:
expected_pairs_per_gender = 68 * 67 // 2
print(f"expected real-prediction rows per gender (68 choose 2): {expected_pairs_per_gender:,}")
print(f"actual: men's {men_mask.sum():,}, women's {women_mask.sum():,}")

assert sub["Pred"].between(0, 1).all(), "found a prediction outside [0, 1]"
print("all predictions within [0, 1]: OK")

seed1_2026 = m_seeds[(m_seeds["Season"] == 2026) & (m_seeds["SeedNum"] == 1)]
seed16_2026 = m_seeds[(m_seeds["Season"] == 2026) & (m_seeds["SeedNum"] == 16)]
if len(seed1_2026) and len(seed16_2026):
    t1, t16 = seed1_2026.iloc[0]["TeamID"], seed16_2026.iloc[0]["TeamID"]
    lo, hi = min(t1, t16), max(t1, t16)
    row = sub[(sub["Team1"] == lo) & (sub["Team2"] == hi)]
    if len(row):
        favored_is_low = t1 == lo
        p_low = row["Pred"].values[0]
        p_favorite = p_low if favored_is_low else 1 - p_low
        print(f"sample 1-seed vs 16-seed check: predicted P(1-seed wins) = {p_favorite:.3f}")

old_sub = pd.read_csv("../submissions/submission.csv")
compare = sub[["ID", "Pred"]].merge(old_sub, on="ID", suffixes=("_v2", "_v1"))
compare["AbsDiff"] = (compare["Pred_v2"] - compare["Pred_v1"]).abs()
real_rows = compare[compare["ID"].isin(sub.loc[has_all_features, "ID"])]
print(f"\nmean |Pred_v2 - Pred_v1| across real-prediction rows: {real_rows['AbsDiff'].mean():.4f}")
print(f"max |Pred_v2 - Pred_v1|: {real_rows['AbsDiff'].max():.4f}")


expected real-prediction rows per gender (68 choose 2): 2,278
actual: men's 2,278, women's 2,278
all predictions within [0, 1]: OK
sample 1-seed vs 16-seed check: predicted P(1-seed wins) = 0.975

mean |Pred_v2 - Pred_v1| across real-prediction rows: 0.0106
max |Pred_v2 - Pred_v1|: 0.0587


## 8. Save the submission file

In [8]:
submission_final = sub[["ID", "Pred"]]
submission_final.to_csv("submission_v2.csv", index=False)
print(f"wrote submission_v2.csv: {len(submission_final):,} rows")
print(submission_final.head(3).to_string(index=False))


wrote submission_v2.csv: 132,133 rows
            ID  Pred
2026_1101_1102   0.5
2026_1101_1103   0.5
2026_1101_1104   0.5


## Conclusion

`submission_v2.csv` was generated using the notebook 18 model (gender-split logistic regression,
base 5 features + `ClusterRankDiff` as a 6th), retrained on all historical tournament games
through 2025 — 2,585 men's and 1,717 women's games, 4,302 total. Clustering was refit on the full
1985-2026 history separately per gender (5 clusters for men's, 8 for women's), so every one of the
136 2026 tournament team-seasons (68 per gender) has a `ClusterRank`.

All 4,556 real tournament-team pairs (2,278 per gender, matching 68-choose-2) got a genuine
prediction, the rest of the 132,133 submission rows default to 0.5, and predictions stayed in a
sane range (0.0008 to 0.9996). The 1-seed vs. 16-seed sanity check moved slightly from the original
submission's 0.973 to 0.975.

**Compared row-by-row against `submission.csv` (the notebook 13/16 model actually submitted to
Kaggle, scored 0.1302747), the average prediction moved by 0.0106 and the largest single-row move
was 0.0587** — a real but modest shift, consistent with a feature that only nudged the walk-forward
score by 0.0006. This file reflects the current best-validated model in the project (0.1669
average walk-forward Brier) and is ready to submit as a second entry.